# Build RAG Vector Database for Bangla LLM Hallucination Detection
This script loads Wikipedia, scraped news, QA trivia, and textbook data, chunks the text,
generates embeddings using the BGE-M3 model on an A100 GPU (with optimal batching),
and builds a compressed FAISS IndexIVFFlat for fast, lightweight retrieval during submission.

## 1. Install Dependencies
Run this cell to install required libraries when running on Kaggle/Colab.
```bash
pip install -q FlagEmbedding faiss-cpu pandas tqdm datasets
```

In [1]:
pip install -q FlagEmbedding faiss-cpu pandas tqdm datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import glob
import re
import pickle
import pandas as pd
import numpy as np
import faiss
import torch
from tqdm import tqdm
from FlagEmbedding import BGEM3FlagModel

## 2. Configuration & Paths
Define all dataset paths and model configuration. Update these paths as needed.

In [3]:
# Input files and directories
# WIKI_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/wikipedia/bangla_wikipedia_top_5.csv"
# NEWS_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/SCRAPING for news/scraped_news.csv"
# QA_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/qa_dataset.csv"
# BOOKS_DIR_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/nctbbook/extracted_text"
# NCTB_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/NEW DATASETS/Textbook Dataset from NCTB.csv"
# # Input files and directories
# WIKI_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/wikipedia/bangla_wikipedia_top_5.csv"
# NEWS_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/SCRAPING for news/scraped_news.csv"
# QA_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/qa_dataset.csv"
# BOOKS_DIR_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/nctbbook/extracted_text"
# NCTB_CSV_PATH = "/home/barnobarno666/KAGGLE/IUT Datathon/DATASET making/NEW DATASETS/Textbook Dataset from NCTB.csv"
# # Input files and directories
WIKI_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-iiwikipedia/bangla_wikipedia_all_rows.csv"
NEWS_CSV_PATH = "ASS/kaggle/input/datasets/barnobarno/rag-corpus-i/scraped_news.csv"
BOOKS_DIR_PATH = "/kaggle/input/datasets/barnobarno/nctb-books-class-9-10-and-grammar-hsc/extracted_text"
NCTB_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-3-0bangla-wo-wiki/Textbook Dataset from NCTB.csv"

# Combined Q&A and New Books directories
COMBINED_QA_CSV_PATH = "/kaggle/input/datasets/barnobarno/non-doc-qna-type-all-combined-for-rag/FINAL_COMBINED.csv"
if not os.path.exists(COMBINED_QA_CSV_PATH):
    COMBINED_QA_CSV_PATH = r"d:\ALL CODES\IUT Datathon\DATASET making\FINAL ONE\FINAL_COMBINED.csv"

NEW_BOOKS_DIR_PATH = "/kaggle/input/datasets/barnobarno/books-nctb-openuni-newer/extracted md oepnuni books+class 8 books"
if not os.path.exists(NEW_BOOKS_DIR_PATH):
    NEW_BOOKS_DIR_PATH = r"d:\ALL CODES\IUT Datathon\RAG IN ACTION\RAG DATA ALL IN\new_extracted_text"

# QA_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/qa_dataset.csvv"
# BAGDHARA_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/bagdhara_dataset.csv"
# GRAMMAR_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/grammer_dataset.csv"
# LEGAL_ACTS_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/bangladesh_legal_acts.csv"
# WORD_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/word_dataset.csv"
# OPEN_TRIVIA_QA_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/open_trivia_qa_dataset.csv"
# OPENBOOKQA_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/openbookqa_dataset.csv"
# SCIQ_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/sciq_dataset.csv"
# SQUAD_V2_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/squad_v2_dataset.csv"
# TRIVIA_QA_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/trivia_qa_dataset.csv"
# NCQA_CSV_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-i/ncqa.csv"


# Output database artifacts
OUTPUT_INDEX_PATH = "bangla_knowledgeFINAL.index"
OUTPUT_METADATA_PATH = "chunks_metadataFINAL.pkl"

# Embedding settings
EMBEDDING_MODEL_ID = "BAAI/bge-m3"
# Increase batch size to 2048 to fully saturate A100 memory & compute
BATCH_SIZE = 2048 
MAX_LENGTH = 512

# Chunking settings
CHUNK_SIZE = 500  # Character length
CHUNK_OVERLAP = 100

## 3. Text Cleaning & Chunking Helpers
Functions to normalize Bangla text and split documents on sentence boundaries.

In [4]:
import unicodedata

def clean_text(text):
    """Normalize whitespace and Unicode representation for correct Bangla conjuncts."""
    if not text or not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFKC', text)
    return " ".join(text.split())

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap_sentences=1):
    """Splits text into overlapping chunks, keeping sentences whole for both start and end boundaries."""
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text] if text else []
    
    # Split on sentence boundary indicators in Bangla (।) and English (. ? !)
    # Do not split if the danda (।) is preceded by 1 to 3 digits (which is typical for section numbers e.g. ১। or ১২।)
    temp = re.sub(r'(?<!\b[0-9\u09E6-\u09EF])(?<!\b[0-9\u09E6-\u09EF]{2})(?<!\b[0-9\u09E6-\u09EF]{3})।\s*', r'।\n', text)
    temp = re.sub(r'([.?!])\s+', r'\1\n', temp)
    sentences = temp.split("\n")
    
    chunks = []
    current_sentences = []
    current_length = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        # If adding this sentence exceeds the chunk size
        if current_length + len(sentence) > chunk_size:
            if current_sentences:
                chunks.append(" ".join(current_sentences))
                # For overlap: keep the last N sentences
                current_sentences = current_sentences[-overlap_sentences:] if len(current_sentences) > overlap_sentences else current_sentences
                current_length = sum(len(s) + 1 for s in current_sentences) - 1 if current_sentences else 0
            
            current_sentences.append(sentence)
            current_length = current_length + len(sentence) + (1 if current_length > 0 else 0)
        else:
            current_sentences.append(sentence)
            current_length += len(sentence) + (1 if len(current_sentences) > 1 else 0)
            
    if current_sentences:
        chunks.append(" ".join(current_sentences))
        
    return chunks

def process_qa_csv(csv_path, source_label, desc_label):
    """Generic helper to load a standard Q&A CSV file and append formatted chunks."""
    if os.path.exists(csv_path):
        print(f"Loading {desc_label} Q&A from {csv_path}...")
        df = pd.read_csv(csv_path, encoding="utf-8")
        print(f"Loaded {len(df)} pairs. Formatting...")
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc_label):
            question = row.get("Question", "")
            answer = row.get("answer", "")
            source = row.get("source", source_label)
            
            if pd.isna(question) or pd.isna(answer):
                continue
                
            qa_text = f"প্রশ্ন: {clean_text(str(question))}\nউত্তর: {clean_text(str(answer))}"
            all_chunks.append({
                "text": qa_text,
                "source": source_label,
                "question": question,
                "answer": answer,
                "dataset_source": source
            })
    else:
        print(f"{desc_label} path {csv_path} not found. Skipping.")

## 4. Loading & Parsing Datasets
Process the four different source datasets into a unified document chunk list.

In [5]:
all_chunks = []

# --- 4.1. Process Wikipedia ---
if os.path.exists(WIKI_CSV_PATH):
    print(f"Loading Wikipedia data from {WIKI_CSV_PATH}...")
    wiki_df = pd.read_csv(WIKI_CSV_PATH, encoding="utf-8")
    print(f"Loaded {len(wiki_df)} Wikipedia articles. Chunking...")
    for _, row in tqdm(wiki_df.iterrows(), total=len(wiki_df), desc="Wiki"):
        content = row.get("content", "")
        title = row.get("title", "")
        url = row.get("url", "")
        wiki_id = row.get("id", "")
        
        if pd.isna(content) or not content:
            continue
            
        chunks = chunk_text(str(content))
        for chunk in chunks:
            all_chunks.append({
                "text": f"শিরোনাম: {title}\nবিষয়বস্তু: {chunk}",
                "source": "wikipedia",
                "title": title,
                "url": url,
                "id": wiki_id
            })
else:
    print(f"Wikipedia path {WIKI_CSV_PATH} not found. Skipping.")

# --- 4.2. Process Scraped News ---
if os.path.exists(NEWS_CSV_PATH):
    print(f"Loading news data from {NEWS_CSV_PATH}...")
    news_df = pd.read_csv(NEWS_CSV_PATH, encoding="utf-8")
    print(f"Loaded {len(news_df)} news articles. Chunking...")
    for _, row in tqdm(news_df.iterrows(), total=len(news_df), desc="News"):
        text = row.get("FullText", "")
        title = row.get("Title", "")
        link = row.get("Link", "")
        date = row.get("Date", "")
        
        if pd.isna(text) or not text:
            continue
            
        chunks = chunk_text(str(text))
        for chunk in chunks:
            all_chunks.append({
                "text": f"তারিখ: {date}\nশিরোনাম: {title}\nপ্রতিবেদন: {chunk}",
                "source": "news",
                "title": title,
                "date": date,
                "link": link
            })
else:
    print(f"News path {NEWS_CSV_PATH} not found. Skipping.")

# --- 4.3. Combined Q&A Dataset ---
if os.path.exists(COMBINED_QA_CSV_PATH):
    print(f"Loading combined Q&A from {COMBINED_QA_CSV_PATH}...")
    combined_df = pd.read_csv(COMBINED_QA_CSV_PATH, encoding="utf-8")
    print(f"Loaded {len(combined_df)} Q&A rows. Formatting...")
    for _, row in tqdm(combined_df.iterrows(), total=len(combined_df), desc="Combined QA"):
        text_val = row.get("text", "")
        source_name = row.get("source", "")
        
        if pd.isna(text_val) or not str(text_val).strip():
            continue
            
        all_chunks.append({
            "text": clean_text(str(text_val)),
            "source": "combined_qa",
            "dataset_source": source_name
        })
else:
    print(f"Combined QA path {COMBINED_QA_CSV_PATH} not found. Skipping.")


# --- 4.4. Process Textbooks ---
if os.path.exists(BOOKS_DIR_PATH):
    print(f"Scanning textbook markdown files in {BOOKS_DIR_PATH}...")
    book_files = glob.glob(os.path.join(BOOKS_DIR_PATH, "*.md")) + glob.glob(os.path.join(BOOKS_DIR_PATH, "*.txt"))
    print(f"Found {len(book_files)} textbook files. Reading & chunking...")
    
    import re
    
    for filepath in tqdm(book_files, desc="Books"):
        filename = os.path.basename(filepath)
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()
            if not content.strip():
                continue
                
            # Clean up markdown headers and separators from the extraction
            # Removes lines like "## Page 1", "## Page 6 (OCR)", and "---"
            content = re.sub(r'(?m)^## Page \d+.*$', '', content)
            content = re.sub(r'(?m)^---$', '', content)
                
            chunks = chunk_text(content)
            for chunk in chunks:
                all_chunks.append({
                    "text": f"উৎস বই: {filename}\nবিষয়বস্তু: {chunk}",
                    "source": "nctb_book",
                    "filename": filename
                })
        except Exception as e:
            print(f"Error reading {filename}: {e}")
else:
    print(f"Books path {BOOKS_DIR_PATH} not found. Skipping.")

# --- 4.4b. Process New Textbooks ---
if os.path.exists(NEW_BOOKS_DIR_PATH):
    print(f"Scanning new textbook markdown files in {NEW_BOOKS_DIR_PATH}...")
    new_book_files = glob.glob(os.path.join(NEW_BOOKS_DIR_PATH, "*.md")) + glob.glob(os.path.join(NEW_BOOKS_DIR_PATH, "*.txt"))
    print(f"Found {len(new_book_files)} new textbook files. Reading & chunking...")
    
    for filepath in tqdm(new_book_files, desc="New Books"):
        filename = os.path.basename(filepath)
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()
            if not content.strip():
                continue
                
            content = re.sub(r'(?m)^## Page \d+.*$', '', content)
            content = re.sub(r'(?m)^---$', '', content)
                
            chunks = chunk_text(content)
            for chunk in chunks:
                all_chunks.append({
                    "text": f"উৎস বই: {filename}\nবিষয়বস্তু: {chunk}",
                    "source": "nctb_book",
                    "filename": filename
                })
        except Exception as e:
            print(f"Error reading {filename}: {e}")
else:
    print(f"New Books path {NEW_BOOKS_DIR_PATH} not found. Skipping.")


# --- 4.5. Process NCTB Textbook CSV ---
if os.path.exists(NCTB_CSV_PATH):
    print(f"Loading NCTB Textbook CSV from {NCTB_CSV_PATH}...")
    nctb_df = pd.read_csv(NCTB_CSV_PATH, encoding="utf-8")
    print(f"Loaded {len(nctb_df)} rows. Formatting as passages...")
    for _, row in tqdm(nctb_df.iterrows(), total=len(nctb_df), desc="NCTB CSV"):
        passage = row.get("Passage", "")
        question = row.get("Question", "")
        answer = row.get("AnsText", "")
        
        if pd.isna(passage) or not passage:
            continue
            
        # Format as a comprehensive passage
        nctb_text = f"তথ্য: {clean_text(str(passage))}"
        if not pd.isna(question) and question and not pd.isna(answer) and answer:
            nctb_text += f"\nপ্রশ্ন: {clean_text(str(question))}\nউত্তর: {clean_text(str(answer))}"
            
        all_chunks.append({
            "text": nctb_text,
            "source": "nctb_csv",
            "question": question,
            "answer": answer
        })
else:
    print(f"NCTB CSV path {NCTB_CSV_PATH} not found. Skipping.")

# Sections 4.6 to 4.15 for individual QA datasets have been commented out as they are unified in Combined Q&A
# (Processed via COMBINED_QA_CSV_PATH in Section 4.3)


print(f"\nTotal generated knowledge base chunks: {len(all_chunks)}")

Loading Wikipedia data from /kaggle/input/datasets/barnobarno/rag-corpus-iiwikipedia/bangla_wikipedia_all_rows.csv...
Loaded 171674 Wikipedia articles. Chunking...


Wiki: 100%|██████████| 171674/171674 [01:53<00:00, 1506.26it/s]


News path ASS/kaggle/input/datasets/barnobarno/rag-corpus-i/scraped_news.csv not found. Skipping.
Loading combined Q&A from /kaggle/input/datasets/barnobarno/non-doc-qna-type-all-combined-for-rag/FINAL_COMBINED.csv...


/tmp/ipykernel_23/40390998.py:58: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df = pd.read_csv(COMBINED_QA_CSV_PATH, encoding="utf-8")


Loaded 285575 Q&A rows. Formatting...


...

Scanning new textbook markdown files in /kaggle/input/datasets/barnobarno/books-nctb-openuni-newer/extracted md oepnuni books+class 8 books...
Found 101 new textbook files. Reading & chunking...


New Books: 100%|██████████| 101/101 [00:06<00:00, 14.62it/s]


Loading NCTB Textbook CSV from /kaggle/input/datasets/barnobarno/rag-corpus-3-0bangla-wo-wiki/Textbook Dataset from NCTB.csv...
Loaded 2802 rows. Formatting as passages...


NCTB CSV: 100%|██████████| 2802/2802 [00:00<00:00, 8471.06it/s]


Total generated knowledge base chunks: 1228615


## 5. GPU-Accelerated Embedding Generation
Initialize BGE-M3 and encode all chunks utilizing the A100 GPU's fp16 acceleration.

In [6]:
import gc

if len(all_chunks) == 0:
    print("Error: No chunks loaded. Cannot build vector database.")
    sys.exit(1)

# Check device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Load BGE-M3
print(f"Loading {EMBEDDING_MODEL_ID}...")
model = BGEM3FlagModel(EMBEDDING_MODEL_ID, use_fp16=(device == "cuda"))

# Extract raw texts
chunk_texts = [item["text"] for item in all_chunks]
num_elements = len(chunk_texts)
dimension = 1024  # BGE-M3 output dimension

Using device: cuda
GPU name: Tesla T4
Loading BAAI/bge-m3...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 6. Initialize FAISS Index
We use IndexFlatIP for exact Inner Product (Cosine Similarity) nearest neighbor search.
This requires no centroid training and yields 100% exact retrieval recall.

In [7]:
print("Preparing FAISS Index...")
index = faiss.IndexFlatIP(dimension)
print(f"FAISS configuration: IndexFlatIP (exact search), dimension={dimension}, size={num_elements}")

Preparing FAISS Index...
FAISS configuration: IndexFlatIP (exact search), dimension=1024, size=1228615


## 7. Batched Embedding and Index Insertion
Process the full corpus in smaller batches (e.g., 100,000 chunks) to keep RAM usage constant and extremely low, avoiding DeadKernelError.

In [8]:
INDEX_BATCH_SIZE = 100000
print(f"Adding all {num_elements} chunks to the index in batches of {INDEX_BATCH_SIZE}...")

for i in range(0, num_elements, INDEX_BATCH_SIZE):
    batch_end = min(i + INDEX_BATCH_SIZE, num_elements)
    print(f"\nProcessing batch {i // INDEX_BATCH_SIZE + 1} (chunks {i} to {batch_end})...")
    
    batch_texts = chunk_texts[i:batch_end]
    batch_outputs = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )
    
    batch_embeddings = batch_outputs["dense_vecs"].astype("float32")
    # Normalize for Cosine Similarity
    norms = np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
    batch_embeddings = batch_embeddings / np.maximum(norms, 1e-12)
    
    print(f"Adding batch embeddings to FAISS index...")
    index.add(batch_embeddings)
    
    # Free memory immediately
    del batch_outputs, batch_embeddings, batch_texts
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nAll embeddings successfully indexed!")

Adding all 1228615 chunks to the index in batches of 100000...

Processing batch 1 (chunks 0 to 100000)...



initial target device: 100%|██████████| 2/2 [00:22<00:00, 11.26s/it]

Inference Embeddings: 100%|██████████| 78/78 [05:59<00:00,  4.61s/it]

Chunks: 100%|██████████| 2/2 [06:39<00:00, 199.95s/it]


Adding batch embeddings to FAISS index...

Processing batch 2 (chunks 100000 to 200000)...


Chunks: 100%|██████████| 2/2 [07:05<00:00, 212.58s/it]


Adding batch embeddings to FAISS index...

Processing batch 3 (chunks 200000 to 300000)...


...

Adding batch embeddings to FAISS index...

Processing batch 12 (chunks 1100000 to 1200000)...


Chunks: 100%|██████████| 2/2 [09:51<00:00, 295.88s/it]


Adding batch embeddings to FAISS index...

Processing batch 13 (chunks 1200000 to 1228615)...


Chunks: 100%|██████████| 2/2 [02:45<00:00, 82.89s/it]


Adding batch embeddings to FAISS index...

All embeddings successfully indexed!


## 7. Serialize Database Artifacts
Save both the FAISS index and the associated text metadata lists to disk.

In [9]:
print(f"Saving FAISS index to {OUTPUT_INDEX_PATH}...")
faiss.write_index(index, OUTPUT_INDEX_PATH)

print(f"Saving chunk metadata to {OUTPUT_METADATA_PATH}...")
with open(OUTPUT_METADATA_PATH, "wb") as f:
    pickle.dump(all_chunks, f)

print("\nOffline Database Build Completed successfully!")
print(f"Artifacts: {OUTPUT_INDEX_PATH} and {OUTPUT_METADATA_PATH}")

Saving FAISS index to bangla_knowledgeFINAL.index...
Saving chunk metadata to chunks_metadataFINAL.pkl...

Offline Database Build Completed successfully!
Artifacts: bangla_knowledgeFINAL.index and chunks_metadataFINAL.pkl
